# Barkley pipe flow - Phase 2: discrete coupled-map-lattice model

The discrete Barkley (2011) model (Eqs. 3-6) is a coupled-map lattice that -
unlike the continuous model - supports stochastic puff **decay** and
**splitting**. **Milestone 1 is implemented** (`discrete.py`): this notebook
reproduces Barkley Fig. 4. The survival statistics (`statistics.py`) are the
remaining milestones - see [`ROADMAP.md`](../ROADMAP.md).


In [ ]:
# --- environment setup (works locally and on Google Colab) ---
try:
    import barkley_pipe  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/yuvrajdabhi2212/barkley-pipe-flow.git"], check=True)
import matplotlib.pyplot as plt
import numpy as np

## The model

$$q_i^{n+1} = F\big(q_i^n + d(q_{i-1}^n - 2q_i^n + q_{i+1}^n),\, u_i^n\big)$$
$$u_i^{n+1} = u_i^n + \varepsilon_1(1-u_i^n) - \varepsilon_2 u_i^n q_i^n - c(u_i^n - u_{i-1}^n)$$

with threshold $\alpha(u,R) = 2000(1-0.8u)/R$ and $F=f^k$ ($k=2$) a
piecewise-linear tent map. $\beta>0$ opens an escape window that makes turbulence
transient (spontaneous decay). $R$ is the discrete Reynolds proxy
($R\approx2000\leftrightarrow Re$).


## The tent map $f(q)$

In [ ]:
from barkley_pipe.discrete import tent_map

q = np.linspace(0.0, 1.8, 600)
for alpha in (0.4, 0.7):
    plt.plot(q, tent_map(q, np.full_like(q, alpha)), label=f"alpha={alpha}")
plt.plot(q, q, "k--", lw=0.8, label="y = q")
plt.xlabel("q"); plt.ylabel("f(q)"); plt.title("Tent map (laminar fixed point at q=0)")
plt.legend(); plt.show()

## Space-time diagrams - reproduction of Barkley Fig. 4

In [ ]:
from barkley_pipe.discrete import DiscreteParams, initial_puff, simulate_discrete
from barkley_pipe.plotting import plot_discrete_spacetime

runs = [(1900, 400, 40, 2400, 10, "decaying puff"),
        (2200, 700, 30, 7000, 28, "puff splitting"),
        (3000, 400, 30, 1600, 6, "expanding slug")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (R, n, w, steps, se, name) in zip(axes, runs):
    q0, u0 = initial_puff(n, width=w, q_level=1.0, seed=0)
    qst = simulate_discrete(q0, u0, DiscreteParams(R=R), n_steps=steps, store_every=se)
    plot_discrete_spacetime(qst, store_every=se, ax=ax, title=f"{name}, R={R}")
plt.tight_layout(); plt.show()

Below onset ($R=1900$) the puff relaminarizes; at $R=2200$ it splits and proliferates; at $R=3000$ a slug fills the domain.

## Remaining: survival statistics (Milestones 2-5)

In [ ]:
from barkley_pipe import statistics
try:
    statistics.fit_tau([1.0, 2.0, 3.0])
except NotImplementedError as e:
    print("statistics layer still a documented stub:", e)

## Plan for the survival statistics (see `ROADMAP.md`)

1. Detect decay/splitting events; Monte-Carlo ensembles (reduced for Colab).
2. Survival fit $P(n)\sim e^{-n/\tau(R)}$ (MLE) -> Fig. 12; decay/split lifetime
   crossing near $R_\times\approx2040$.
3. Turbulence fraction $F_t(R)$, onset $R_c\approx2046$, exponent
   $\beta_{DP}\approx0.2765$.

**Citation note:** for puff-*splitting* mechanisms cite Svirsky, Grafke &
Frishman (2025, arXiv:2505.05075) and Frishman & Grafke (2022, arXiv:2205.05578);
the Moron-Vela-Martin-Avila (2025) JFM paper concerns decay *predictability*.
